In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [14]:
df = pd.read_csv('covid_toy.csv')
df.sample(5)

,age,gender,fever,cough,city,has_covid
71,75,Female,104.0,Strong,Delhi,No
59,6,Female,104.0,Mild,Kolkata,Yes
48,66,Male,99.0,Strong,Bangalore,No
85,16,Female,103.0,Mild,Bangalore,Yes
5,84,Female,NaN,Mild,Bangalore,Yes


In [15]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [16]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)
X_train.shape,X_test.shape

((80, 5), (20, 5))

In [17]:
X_train

,age,gender,fever,cough,city
57,49,Female,99.0,Strong,Bangalore
43,22,Female,99.0,Mild,Bangalore
81,65,Male,99.0,Mild,Delhi
41,82,Male,NaN,Mild,Kolkata
89,46,Male,103.0,Strong,Bangalore
...,...,...,...,...,...
0,60,Male,103.0,Mild,Kolkata
94,79,Male,NaN,Strong,Kolkata
75,5,Male,102.0,Mild,Kolkata
47,18,Female,104.0,Mild,Bangalore


# Normal PIPELINE:

In [18]:
# Adding SimpleImputer to handle missing values in the dataset. We will use the 'most_frequent' strategy for categorical features and 'mean' strategy for numerical features.
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.transform(X_test[['fever']])


In [19]:
X_train_fever = pd.DataFrame(X_train_fever,columns=['fever'])
X_test_fever = pd.DataFrame(X_test_fever,columns=['fever'])

In [20]:
X_train_fever.shape,X_test_fever.shape

((80, 1), (20, 1))

In [22]:
# OrdinalEncoding is used to convert categorical features into numerical values. It assigns a unique integer to each category in the feature. This is useful for machine learning algorithms that require numerical input.
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])
X_train_cough.shape,X_test_cough.shape

((80, 1), (20, 1))

In [23]:
X_train_cough = pd.DataFrame(X_train_cough,columns=['cough'],dtype=int)
X_test_cough = pd.DataFrame(X_test_cough,columns=['cough'],dtype=int)

In [24]:
X_train_cough

,cough
0,1
1,0
2,0
3,0
4,1
...,...
75,0
76,1
77,0
78,0


In [26]:
# OneHotEncoding is used to convert categorical features into a binary matrix representation. Each category is represented as a separate binary feature, where 1 indicates the presence of that category and 0 indicates its absence. This is useful for machine learning algorithms that require numerical input and do not assume any ordinal relationship between categories.
ohe = OneHotEncoder(sparse_output=False,drop='first')
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])
X_test_gender_city = ohe.transform(X_test[['gender','city']])
X_train_gender_city.shape

(80, 4)

In [27]:
X_train_gender_city = pd.DataFrame(X_train_gender_city,columns=ohe.get_feature_names_out(['gender', 'city']),dtype=int)
X_train_gender_city

,gender_Male,city_Delhi,city_Kolkata,city_Mumbai
0,0,0,0,0
1,0,0,0,0
2,1,1,0,0
3,1,0,1,0
4,1,0,0,0
...,...,...,...,...
75,1,0,1,0
76,1,0,1,0
77,1,0,1,0
78,0,0,0,0


In [28]:
#Extracting Age:
X_train_age= X_train.drop(columns=['gender','fever','cough','city']).values
X_test_age= X_test.drop(columns=['gender','fever','cough','city']).values
X_train_age.shape, X_test_age.shape

((80, 1), (20, 1))

In [29]:
# Concatenate All:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough), axis=1)
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough), axis=1)
X_train_transformed.shape, X_test_transformed.shape

((80, 7), (20, 7))

In [30]:
column_names=['age','fever','gender_Male','city_Delhi','city_Kolkata','city_Mumbai','cough']
X_train_transformed = pd.DataFrame(X_train_transformed, columns=column_names)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=column_names)

In [31]:
X_train_transformed

,age,fever,gender_Male,city_Delhi,city_Kolkata,city_Mumbai,cough
0,49.0,99.000000,0.0,0.0,0.0,0.0,1.0
1,22.0,99.000000,0.0,0.0,0.0,0.0,0.0
2,65.0,99.000000,1.0,1.0,0.0,0.0,0.0
3,82.0,100.535211,1.0,0.0,1.0,0.0,0.0
4,46.0,103.000000,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...
75,60.0,103.000000,1.0,0.0,1.0,0.0,0.0
76,79.0,100.535211,1.0,0.0,1.0,0.0,1.0
77,5.0,102.000000,1.0,0.0,1.0,0.0,0.0
78,18.0,104.000000,0.0,0.0,0.0,0.0,0.0


# USING COLUMN TRANSFORMER :

In [32]:
from sklearn.compose import ColumnTransformer

In [34]:

transformer= ColumnTransformer(transformers=[
    ('t1',SimpleImputer(),['fever']),
    ('t2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('t3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city']) 
], remainder='passthrough')

In [35]:
transformer.fit_transform(X_train).shape

(80, 7)

In [36]:
transformer.transform(X_test).shape

(20, 7)